# 1. Sen4AgriNet: prepare references and collect annual imagery

Carrot and potato references come from declared crop-mask cells. Cabbage is supplied by FranceCrops. This is a bounded subset, not a full-release audit.


## 1. Setup and configuration

Choose a Python 3.12+ kernel, review the settings below, then **Restart Kernel → Run All**. Opening the notebook starts no work. Existing settings, selections, archives and jobs are preserved.

- `reuse`: use local references; satellite collection is enabled separately.
- `download_missing`: obtain missing references within the source limits.
- `offline`: use local references, with no package installation, sign-in or satellite calls. Dependencies must already be installed.

Set `extra_batches` to the number of additional batches to collect before 05–07. Each batch can select up to 50 unused parcels per available crop. Keep the same value to resume or increase it to add batches.

Use `download_missing` on a fresh clone. Enter an Earth Engine-enabled project ID when prompted. All sources target **up to 50 candidate parcels per available crop per round**; existing larger batches remain intact. Saved outputs may describe earlier runs until cells are rerun.


In [ ]:
# Review these settings before Run All.
reference_mode = "reuse"  # reuse | download_missing | offline
run_satellite_collection = True  # offline always disables satellite calls
submit_only = True  # Queue jobs now. Set False to collect finished results later.

from pathlib import Path
import os
import sys
repository_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), Path.cwd())
source_folder = str(repository_root / "src")
if source_folder not in sys.path:
    sys.path.insert(0, source_folder)
from crop_pipeline import notebook_setup, source_workflow
from crop_pipeline.notebook_display import show_table, show_mapping, show_items
reference_options = source_workflow.reference_options(reference_mode)
notebook_setup.prepare_environment(repository_root, install_missing=reference_mode != "offline")
os.chdir(repository_root)

from crop_pipeline.sources import adaptive, annual
from crop_pipeline import progress as pipeline_progress
from crop_pipeline.windows.seasons import file_sha256
settings_path = Path("settings.local.json")
local_settings = notebook_setup.ensure_local_settings(settings_path)
report_folder = Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).expanduser().resolve()
annual_archive_root = annual.archive_root(local_settings, report_folder)
run_name = adaptive.COLLECTION_NAME
candidate_parcels_per_crop = source_workflow.ROUND_PARCELS_PER_CROP
extra_batches = 0  # Number of additional batches this round. Keep the value when resuming.
run_satellite_collection = run_satellite_collection and reference_mode != "offline"
authenticate_now = run_satellite_collection
download_workers = 2
intervals_per_request = 6

from crop_pipeline.sources import sen4agrinet as sen4agrinet_references, bounded_references
import json
source_name = "sen4agrinet"
inward_buffer_metres = 0  # Preserve declared crop-mask cells; no additional erosion.
dataset_folder = Path(os.environ.get("S4A_ROOT", local_settings.get("dataset_root", "./ground_truth"))).expanduser().resolve()
reference_root = Path(local_settings.get("sen4agrinet_reference_root", str(annual_archive_root / "sen4agrinet_references"))).expanduser().resolve()
reference_target_crops = ("carrot", "potato")
reference_target_parcels_per_crop = 100  # Reference pool budget, separate from the round limit.
reference_scan_max_patches = 350  # Explicit cap; increase deliberately to expand the pool.
dataset_revision = None  # Reuse the first pinned release.

# Resolve account configuration before source transfers; authentication occurs during collection.
earth_engine_project = notebook_setup.project_for_collection(local_settings, settings_path, required=run_satellite_collection)
print(f"Source: {source_name} | References: {reference_mode} | Round limit: {candidate_parcels_per_crop} per crop")
print(f"Inward buffer: {inward_buffer_metres} m | Satellite collection: {run_satellite_collection}")
print("Archive:", annual_archive_root)
print("Backend: Earth Engine batch assets → Parquet transport → annual TIFFs")
notebook_progress = pipeline_progress.start_notebook(
    '01_sen4agrinet_preparation', source_workflow.STAGES, report_folder.parent / "progress",
    workload={"folder": str(annual_archive_root), "source": source_name},
)


## 2. Obtain references

Reuse the pinned release and saved mask checkpoints. In `download_missing` mode, read only label/parcel masks and coordinates using HTTP byte ranges; no full imagery NetCDF downloads. Stop at the reference target or patch cap and report available candidates and deficits.

To expand an exhausted pool, deliberately increase `reference_target_parcels_per_crop` and, if needed, `reference_scan_max_patches`, then use `download_missing`. Saved selections and jobs stay intact. Unseen patches have not been audited.


In [3]:
pipeline_progress.begin_stage("Obtain references")
repository_manifest = sen4agrinet_references.discover_release(
    reference_root, enabled=reference_options["download_if_missing"], revision=dataset_revision,
)
reference_summary = bounded_references.prepare(
    repository_manifest, reference_root, dataset_folder,
    parcels_per_crop=reference_target_parcels_per_crop, seed=adaptive.DEFAULT_POLICY["seed"],
    download_enabled=reference_options["download_if_missing"], scan_enabled=True,
    target_crops=reference_target_crops, max_patches=reference_scan_max_patches,
    cached_only=reference_options["cached_only"],
)
reference_note = f"{reference_summary.get('scope', 'verified_full_release')}; {reference_summary['files_scanned']} patches; {reference_summary.get('stop_reason', 'full_inventory_reused')}"
print("Pinned release:", repository_manifest["revision"])
print("Reference coverage:", reference_note)
show_mapping(reference_summary.get("deficits", {}), title="Reference deficits", value_label="Parcels still needed")
pipeline_progress.end_stage()


01_sen4agrinet_preparation: stages:  40%|████      | 2/5 [00:01<00:02,  1.32stage/s, elapsed 0m 01s; notebook ETA learning (future stages not yet timed)]                    

Pinned dataset release: 4eb44157012b9ec251767721d6bae55cf0c564e2
NetCDF patches: 10013
Years: [2019, 2020]
Year/tile combinations: 16
Full NetCDF size, NOT the planned transfer (GiB): 261.75
Largest upstream NetCDF, NOT downloaded whole (GiB): 0.06
Reference run: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_archive/sen4agrinet_references/releases/4eb44157012b9ec251767721d6bae55cf0c564e2/inventories/4710076785993f11


01_sen4agrinet_preparation: stages:  60%|██████    | 3/5 [00:01<00:01,  1.95stage/s, elapsed 0m 01s; notebook ETA learning (future stages not yet timed)]                 

Reusing bounded reference inventory: 350 patches.
Reference scope: bounded_training_reference_subset
Reference stop reason: patch_cap_reached
Reference deficits: {'carrot': 215, 'potato': 0}
Cabbage source: FranceCrops (not requested from Sen4AgriNet)
New reference patch transfers enabled: False
Reference scan: complete
Files scanned: 350
Eligible crop counts: {'carrot': {'fragments': 36, 'parcel_groups': 35}, 'potato': {'fragments': 702, 'parcel_groups': 698}}
Excluded ambiguous parcel groups: 0
Scanned reference fields: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_archive/sen4agrinet_references/releases/4eb44157012b9ec251767721d6bae55cf0c564e2/inventories/4710076785993f11/bounded/710faa22469fd827/fields
Reference bytes received this scan (MiB): 0.0
Patches reused from checkpoints: 350


## 3. Validate references

Load the verified reference inventory. Preserve crop-code, coordinate/grid, geometry, identity and ambiguous-group checks. Sen4AgriNet retains its declared mask cells: **0 m additional inward buffer**, as documented in the methodology.


In [ ]:
pipeline_progress.begin_stage("Validate references")
reference_features, inventory_provenance = annual.load_sen4agrinet_references(Path(reference_summary["field_root"]))
show_mapping(reference_summary["crop_counts"], title="Eligible reference parcels and pixels")
print("Excluded ambiguous groups:", reference_summary["excluded_parcel_groups"])
print("Edge handling: retain declared crop-mask cells (0 m additional buffer).")
pipeline_progress.end_stage()


## 4. Prepare the collection round

Resume original saved plans and jobs. Fill the round limit only with previously unselected parcels for deficient crop/partition combinations; locked partitions stay unchanged. Prepare masks only for selected candidates and show the combined workload. Source-specific edge handling is preserved.


In [ ]:
pipeline_progress.begin_stage("Prepare collection round")
adaptive_batches = adaptive.prepare_round_batches(
    "sen4agrinet", reference_features, annual_archive_root, report_folder,
    provenance={"dataset": sen4agrinet_references.DATASET_NAME,
                "dataset_revision": repository_manifest["revision"],
                "coverage": reference_summary.get("scope", "verified_full_release"),
                "reference_stop_reason": reference_summary.get("stop_reason", "full_inventory_reused"),
                "reference_scanned_files": reference_summary.get("scanned_files", []),
                "repository_files_scanned": reference_summary["files_scanned"],
                "reference_summary": str(Path(reference_summary["field_root"]).parent / "completed_inventory.json"),
                "inventories": inventory_provenance},
    inward_buffer_metres=inward_buffer_metres, parcels_per_crop=candidate_parcels_per_crop, extra_batches=extra_batches,
    candidate_crops=reference_target_crops,
)
annual_plan, adaptive_context = next(((plan, context) for plan, context in adaptive_batches if plan is not None), (None, adaptive_batches[-1][1]))
print("Active/resumed batches:", len(adaptive_batches))
print("Selected parcels:", sum(context["candidate_parcels"] for plan, context in adaptive_batches if plan is not None))
print("Tiles:", sum(plan["estimate"]["tiles"] for plan, context in adaptive_batches if plan is not None))
show_items(sorted({year for plan, context in adaptive_batches if plan is not None for year in plan["years"]}), title="Years")
pipeline_progress.set_workload({"folder": str(annual_archive_root), "source": source_name,
    "plans": [plan["revision"] for plan, context in adaptive_batches if plan is not None],
    "satellite_collection": run_satellite_collection})
pipeline_progress.end_stage()


## 5. Collect annual imagery

Submit or collect existing Earth Engine jobs. Keep the laptop on until Run All returns normally. Submitted jobs continue in the cloud; rerun later to retrieve results and create annual TIFFs.

Cloud masks, daily mosaics, bilinear band alignment and half-month medians are unchanged. Each completed tile retains 24 periods × seven measurements. One progress display follows the active step; detailed task/page messages are saved to its log. Completion describes saved data, not usable-window quality.


In [ ]:
pipeline_progress.begin_stage("Collect annual imagery")
collection_result = adaptive.collect_batches(
    adaptive_batches, earth_engine_project,
    enabled=run_satellite_collection, authenticate=authenticate_now,
    submit_only=submit_only,
    download_workers=download_workers, intervals_per_request=intervals_per_request,
)
pipeline_progress.end_stage("skipped" if not run_satellite_collection else "completed" if collection_result["status"] == "completed" else "pending")


## 6. Summary and next action

Check parcel counts, verified tiles, and cloud job states below. Notebook 04 can collect JECAM in parallel. Run [Notebook 05](05_external_imagery_quality.ipynb) after the completed development and JECAM batches are available locally.

Annual TIFFs are under `annual_tifs/<source>/<year>/<campaign>/round_<number>/`, beside the configured archive root. Masks and Parquet caches remain separate. See [the workflow guide](../docs/workflow.md).


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
source_workflow.print_summary(
    source_name, adaptive_batches, collection_result, reference_mode,
    inward_buffer_metres, reference_note=reference_note,
)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
